**Token Classification: POS Tagging & Chunking**

**Objective:** Build a token classification system using BERT to perform POS Tagging and Chunking.

**Model:** `bert-base-uncased` via Hugging Face Transformers

**Dataset:** CoNLL-2003

---

### Pipeline
```
Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison
```

## Install Required Libraries

In [ ]:
!pip uninstall -y datasets
!pip install datasets==2.19.1 transformers seqeval

In [ ]:
!pip install transformers datasets torch seqeval scikit-learn accelerate -q

In [ ]:
!pip install evaluate -q

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate

---
## Task 1: Dataset Selection (10%)

### Dataset: CoNLL-2003

CoNLL-2003 is a widely used benchmark dataset for sequence labeling tasks. It contains:
- **tokens** – words in each sentence
- **pos_tags** – Part-of-Speech labels
- **chunk_tags** – Chunking labels in BIO format
- **ner_tags** – Named Entity Recognition labels (not used here)

### Label Types

**POS Tags** label each word's grammatical role:
| Tag | Meaning | Example |
|---|---|---|
| NN | Noun (singular) | cat, dog |
| NNS | Noun (plural) | cats, dogs |
| VB | Verb (base form) | run, eat |
| VBZ | Verb (3rd person) | runs, eats |
| JJ | Adjective | big, fast |
| DT | Determiner | the, a |
| IN | Preposition | in, on, at |
| NNP | Proper noun | John, Google |

**Chunk Tags** group words into phrases using BIO format:
| Tag | Meaning |
|---|---|
| B-NP | Beginning of a Noun Phrase |
| I-NP | Inside a Noun Phrase |
| B-VP | Beginning of a Verb Phrase |
| I-VP | Inside a Verb Phrase |
| B-PP | Beginning of a Prepositional Phrase |
| O | Outside any phrase |

In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003")

In [ ]:
# Load CoNLL-2003 dataset
dataset = load_dataset('conll2003')

print("Dataset Structure:")
print(dataset)

# Extract label lists
pos_label_list   = dataset['train'].features['pos_tags'].feature.names
chunk_label_list = dataset['train'].features['chunk_tags'].feature.names

print(f"\nPOS Tag Labels   ({len(pos_label_list)}): {pos_label_list}")
print(f"\nChunk Tag Labels ({len(chunk_label_list)}): {chunk_label_list}")

In [ ]:
# Preview a sample
import matplotlib.pyplot as plt


sample = dataset['train'][0]
print("Sample Sentence:")
print(f"Tokens     : {sample['tokens']}")
print(f"POS Tags   : {[pos_label_list[i]   for i in sample['pos_tags']]}")
print(f"Chunk Tags : {[chunk_label_list[i] for i in sample['chunk_tags']]}")

# Visualize label distribution
all_pos_ids   = [tag for ex in dataset['train'] for tag in ex['pos_tags']]
all_chunk_ids = [tag for ex in dataset['train'] for tag in ex['chunk_tags']]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

pos_counts = pd.Series([pos_label_list[i] for i in all_pos_ids]).value_counts()
pos_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('POS Tag Distribution', fontweight='bold')
axes[0].set_xlabel('POS Tag')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

chunk_counts = pd.Series([chunk_label_list[i] for i in all_chunk_ids]).value_counts()
chunk_counts.plot(kind='bar', ax=axes[1], color='#2ecc71', edgecolor='black')
axes[1].set_title('Chunk Tag Distribution', fontweight='bold')
axes[1].set_xlabel('Chunk Tag')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150)
plt.show()

---
## Task 2: Data Preprocessing (15%)

### The Subword Challenge

BERT uses **WordPiece tokenization** which splits words into sub-tokens:

```
Word       : "playing"
BERT splits: ["play", "##ing"]  ← 2 tokens from 1 word
```

The original label was for the whole word `playing` but now BERT sees 2 tokens. We must:
- Assign the label to the **first sub-token**
- Assign **-100** to all other sub-tokens (PyTorch ignores -100 during loss calculation)
- Assign **-100** to special tokens `[CLS]` and `[SEP]` too

In [ ]:
MODEL_NAME = 'bert-base-uncased'
MAX_LEN    = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")

# Demonstrate subword tokenization
example_words  = ["John", "is", "playing", "football"]
example_tokens = tokenizer(example_words, is_split_into_words=True)
print(f"\nOriginal words : {example_words}")
print(f"Token IDs      : {example_tokens['input_ids']}")
print(f"Tokens         : {tokenizer.convert_ids_to_tokens(example_tokens['input_ids'])}")
print(f"Word IDs       : {example_tokens.word_ids()}")
print("\nNotice: 'playing' becomes ['play', '##ing'] → word_id stays the same (2)")

In [ ]:
def tokenize_and_align_labels(examples, label_column):
    """
    Tokenize words and align labels to sub-tokens.

    Rules:
    - First sub-token of a word  → gets the real label
    - Other sub-tokens of a word → get -100 (ignored in loss)
    - Special tokens [CLS][SEP]  → get -100 (ignored in loss)

    Returns:
    - input_ids      : token indices
    - attention_mask : 1 for real tokens, 0 for padding
    - labels         : aligned label ids
    """
    tokenized_inputs = tokenizer(
        examples['tokens'],
        is_split_into_words=True,
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN
    )

    all_labels = []

    for i, labels in enumerate(examples[label_column]):
        word_ids      = tokenized_inputs.word_ids(batch_index=i)
        label_ids     = []
        previous_word = None

        for word_id in word_ids:
            if word_id is None:
                # Special tokens [CLS] and [SEP] → ignore
                label_ids.append(-100)
            elif word_id != previous_word:
                # First sub-token of a word → assign real label
                label_ids.append(labels[word_id])
            else:
                # Subsequent sub-tokens → ignore
                label_ids.append(-100)

            previous_word = word_id

        all_labels.append(label_ids)

    tokenized_inputs['labels'] = all_labels
    return tokenized_inputs


# Use a small subset for faster training on local/CPU
# Remove .select() for full training on Colab GPU
small_train = dataset['train'].select(range(1500))
small_val   = dataset['validation'].select(range(300))
small_test  = dataset['test'].select(range(300))

# ── POS Tagging Datasets ──
pos_train_data = small_train.map(lambda x: tokenize_and_align_labels(x, 'pos_tags'), batched=True)
pos_val_data   = small_val.map(lambda x:   tokenize_and_align_labels(x, 'pos_tags'), batched=True)
pos_test_data  = small_test.map(lambda x:  tokenize_and_align_labels(x, 'pos_tags'), batched=True)

# ── Chunking Datasets ──
chunk_train_data = small_train.map(lambda x: tokenize_and_align_labels(x, 'chunk_tags'), batched=True)
chunk_val_data   = small_val.map(lambda x:   tokenize_and_align_labels(x, 'chunk_tags'), batched=True)
chunk_test_data  = small_test.map(lambda x:  tokenize_and_align_labels(x, 'chunk_tags'), batched=True)

print("Tokenization and label alignment complete!")
print(f"\nPOS Train size   : {len(pos_train_data)}")
print(f"Chunk Train size : {len(chunk_train_data)}")

# Show sample output
print(f"\nSample input_ids      (first 10): {pos_train_data[0]['input_ids'][:10]}")
print(f"Sample attention_mask (first 10): {pos_train_data[0]['attention_mask'][:10]}")
print(f"Sample labels         (first 10): {pos_train_data[0]['labels'][:10]}")
print("\n(-100 = special/sub-token → ignored during training)")

---
## Task 3: Model Setup (15%)

We use `AutoModelForTokenClassification` which adds a **linear classification layer** on top of BERT.
For each token, BERT outputs a vector → the linear layer maps it to a label.

In [ ]:
# ── Label Mappings ──
# POS
pos_id2label = {i: label for i, label in enumerate(pos_label_list)}
pos_label2id = {label: i for i, label in enumerate(pos_label_list)}

# Chunking
chunk_id2label = {i: label for i, label in enumerate(chunk_label_list)}
chunk_label2id = {label: i for i, label in enumerate(chunk_label_list)}

NUM_POS_LABELS   = len(pos_label_list)
NUM_CHUNK_LABELS = len(chunk_label_list)

print(f"POS Labels   : {NUM_POS_LABELS}")
print(f"Chunk Labels : {NUM_CHUNK_LABELS}")

# ── Load POS Model ──
pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_POS_LABELS,
    id2label=pos_id2label,
    label2id=pos_label2id
)
print(f"\nPOS Model loaded | Trainable params: {sum(p.numel() for p in pos_model.parameters() if p.requires_grad):,}")

# ── Load Chunking Model ──
chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CHUNK_LABELS,
    id2label=chunk_id2label,
    label2id=chunk_label2id
)
print(f"Chunk Model loaded | Trainable params: {sum(p.numel() for p in chunk_model.parameters() if p.requires_grad):,}")

---
## Task 4: Training using Hugging Face Trainer (20%)

In [ ]:
# ── seqeval metric for evaluation during training ──
seqeval = evaluate.load('seqeval')

def make_compute_metrics(id2label):
    """Factory function that returns a compute_metrics function for a given id2label map."""
    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        predictions    = np.argmax(logits, axis=-1)

        true_labels = [
            [id2label[l] for l in label_seq if l != -100]
            for label_seq in labels
        ]
        true_preds = [
            [id2label[p] for p, l in zip(pred_seq, label_seq) if l != -100]
            for pred_seq, label_seq in zip(predictions, labels)
        ]

        results = seqeval.compute(predictions=true_preds, references=true_labels)
        return {
            'precision': results['overall_precision'],
            'recall'   : results['overall_recall'],
            'f1'       : results['overall_f1'],
            'accuracy' : results['overall_accuracy']
        }
    return compute_metrics


# Data collator handles dynamic padding per batch
data_collator = DataCollatorForTokenClassification(tokenizer)

print("Metric and data collator ready.")

In [ ]:
# ─────────────────────────────────────────────
#  TRAIN POS TAGGING MODEL
# ─────────────────────────────────────────────

pos_training_args = TrainingArguments(
    output_dir                  = './pos_model',
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    logging_steps               = 50,
    report_to                   = 'none'     # Disable wandb
)

pos_trainer = Trainer(
    model           = pos_model,
    args            = pos_training_args,
    train_dataset   = pos_train_data,
    eval_dataset    = pos_val_data,
    processing_class       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = make_compute_metrics(pos_id2label)
)

print("Starting POS Tagging Training...\n")
pos_trainer.train()
print("\nPOS Tagging Training Complete!")

In [ ]:
# ─────────────────────────────────────────────
#  TRAIN CHUNKING MODEL
# ─────────────────────────────────────────────

chunk_training_args = TrainingArguments(
    output_dir                  = './chunk_model',
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    logging_steps               = 50,
    report_to                   = 'none'
)

chunk_trainer = Trainer(
    model           = chunk_model,
    args            = chunk_training_args,
    train_dataset   = chunk_train_data,
    eval_dataset    = chunk_val_data,
    processing_class       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = make_compute_metrics(chunk_id2label)
)

print("Starting Chunking Training...\n")
chunk_trainer.train()
print("\nChunking Training Complete!")

---
## Task 5: Evaluation using seqeval (15%)

**seqeval** evaluates at the **phrase/entity level** — not just individual tokens.
A phrase is counted correct only if ALL its tokens are correctly labeled.

In [ ]:
def full_evaluate(trainer, test_data, id2label, task_name):
    """Run evaluation on test set and print detailed metrics."""
    predictions, labels, _ = trainer.predict(test_data)
    preds = np.argmax(predictions, axis=-1)

    true_labels = [
        [id2label[l] for l in label_seq if l != -100]
        for label_seq in labels
    ]
    true_preds = [
        [id2label[p] for p, l in zip(pred_seq, label_seq) if l != -100]
        for pred_seq, label_seq in zip(preds, labels)
    ]

    results = seqeval.compute(predictions=true_preds, references=true_labels)

    print(f"\n{'='*45}")
    print(f" {task_name} – Test Set Results")
    print(f"{'='*45}")
    print(f"  Precision : {results['overall_precision']:.4f}")
    print(f"  Recall    : {results['overall_recall']:.4f}")
    print(f"  F1 Score  : {results['overall_f1']:.4f}")
    print(f"  Accuracy  : {results['overall_accuracy']:.4f}")

    return results, true_labels, true_preds


# Evaluate both models
pos_results,   pos_true,   pos_pred   = full_evaluate(pos_trainer,   pos_test_data,   pos_id2label,   'POS Tagging')
chunk_results, chunk_true, chunk_pred = full_evaluate(chunk_trainer, chunk_test_data, chunk_id2label, 'Chunking')

In [ ]:
# Plot per-label F1 scores for POS tagging
pos_per_label = {k: v for k, v in pos_results.items() if isinstance(v, dict)}

if pos_per_label:
    labels_plot = list(pos_per_label.keys())
    f1_scores   = [pos_per_label[l]['f1'] for l in labels_plot]

    plt.figure(figsize=(14, 5))
    plt.bar(labels_plot, f1_scores, color='steelblue', edgecolor='black')
    plt.title('POS Tagging – Per Label F1 Score', fontweight='bold')
    plt.xlabel('POS Tag')
    plt.ylabel('F1 Score')
    plt.xticks(rotation=45)
    plt.ylim(0, 1.1)
    plt.tight_layout()
    plt.savefig('POS_per_label_F1.png', dpi=150)
    plt.show()

---
## Task 6: Inference on Custom Sentences (10%)

**Example Input:** `John works at Google in California`  
**Expected Output:** POS Tags + Chunk Tags for each word

In [ ]:
import torch
print("torch loaded!")

In [ ]:
def predict_tags(sentence, model, tokenizer, id2label):
    model.eval()
    words  = sentence.split()
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=MAX_LEN
    )

    # ✅ Move inputs to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    preds    = torch.argmax(outputs.logits, dim=-1).squeeze(0)
    word_ids = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LEN
    ).word_ids()

    results      = []
    seen_word_id = set()

    for idx, word_id in enumerate(word_ids):
        if word_id is not None and word_id not in seen_word_id:
            results.append((words[word_id], id2label[preds[idx].item()]))
            seen_word_id.add(word_id)

    return results

---
## Task 7: Comparison – POS Tagging vs Chunking (10%)

In [ ]:
# ── Comparison Table ──
comparison_df = pd.DataFrame({
    'Task'      : ['POS Tagging', 'Chunking'],
    'Difficulty': ['Easy – grammar-level tagging', 'Medium – phrase-level grouping'],
    'Precision' : [round(pos_results['overall_precision'], 4), round(chunk_results['overall_precision'], 4)],
    'Recall'    : [round(pos_results['overall_recall'],    4), round(chunk_results['overall_recall'],    4)],
    'F1 Score'  : [round(pos_results['overall_f1'],        4), round(chunk_results['overall_f1'],        4)],
    'Accuracy'  : [round(pos_results['overall_accuracy'],  4), round(chunk_results['overall_accuracy'],  4)]
})

print("\n" + "="*75)
print(" COMPARISON: POS TAGGING vs CHUNKING")
print("="*75)
print(comparison_df.to_string(index=False))
print("="*75)

# Bar chart comparison
metrics_names = ['Precision', 'Recall', 'F1 Score', 'Accuracy']
pos_vals   = [pos_results['overall_precision'],   pos_results['overall_recall'],   pos_results['overall_f1'],   pos_results['overall_accuracy']]
chunk_vals = [chunk_results['overall_precision'], chunk_results['overall_recall'], chunk_results['overall_f1'], chunk_results['overall_accuracy']]

x = np.arange(len(metrics_names))
w = 0.3

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, pos_vals,   w, label='POS Tagging', color='#3498db', edgecolor='black')
b2 = ax.bar(x + w/2, chunk_vals, w, label='Chunking',    color='#2ecc71', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('POS Tagging vs Chunking – Performance Comparison', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bars in [b1, b2]:
    for bar in bars:
        ax.annotate(f'{bar.get_height():.3f}',
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('POS_vs_Chunking_Comparison.png', dpi=150)
plt.show()

---
## Task 8: Report / Blog (5%)

### POS Tagging vs Chunking – Differences, Challenges & Observations

---

### What is POS Tagging?
POS (Part-of-Speech) Tagging assigns a grammatical label to each individual word in a sentence. For example:
- `"John"` → NNP (Proper Noun)
- `"runs"` → VBZ (Verb, 3rd person singular)
- `"fast"` → RB (Adverb)

Each token is independently labeled based on its grammatical role. It answers: *"What type of word is this?"*

---

### What is Chunking?
Chunking (also called Shallow Parsing) groups words into meaningful phrases using BIO (Beginning-Inside-Outside) format:
- `B-NP` → Beginning of a Noun Phrase
- `I-NP` → Inside a Noun Phrase (continuation)
- `O`    → Outside any phrase

Example: `"The big cat"` → `B-NP I-NP I-NP`

It answers: *"Which words belong to the same phrase?"*

---

### Key Differences

| Aspect | POS Tagging | Chunking |
|---|---|---|
| Level | Word level | Phrase level |
| Complexity | Easier | Harder |
| Label format | Flat (NN, VB, JJ) | BIO (B-NP, I-NP, O) |
| What it identifies | Grammar of each word | Groupings of words into phrases |
| Example output | `cat → NN` | `The big cat → B-NP I-NP I-NP` |

---

### Challenges Faced

1. **Sub-word Tokenization (Label Alignment)**
   BERT splits words like `"playing"` into `["play", "##ing"]`. The original label was for the whole word, but BERT sees 2 tokens. Careful alignment using `-100` for sub-tokens was required.

2. **BIO Consistency in Chunking**
   Chunking requires the model to understand phrase boundaries — it is not enough to get individual tags right. A phrase is evaluated correctly only if every token in it is labeled correctly.

3. **Compute on CPU**
   BERT is a large model (~110M parameters). Training on CPU is slow, so a smaller subset of the dataset was used for faster experimentation.

---

### Observations & Insights

1. **BERT excels at POS Tagging** because its pre-trained representations already encode strong grammatical information from reading massive text corpora.

2. **Chunking is harder** — the model needs to learn both the phrase type AND phrase boundaries simultaneously, making it a more complex task.

3. **Label Alignment is critical** — incorrect alignment of sub-token labels leads to wrong training signals and poor model performance.

4. **seqeval vs token-level accuracy** — seqeval evaluates at the phrase level which is stricter. A model can have high token accuracy but lower seqeval F1 if phrase boundaries are slightly wrong.

5. **Hugging Face Trainer** significantly simplifies the training loop — handling evaluation, checkpointing, and logging automatically.

---

### Conclusion

Fine-tuning BERT for token classification tasks like POS Tagging and Chunking demonstrates the power of transfer learning in NLP. Even with a small training dataset, BERT achieves strong performance by leveraging its pre-trained language understanding. POS Tagging is more straightforward while Chunking requires the model to capture phrase-level structure, making it a better test of deep language understanding.